# 📋 HOJA DE APUNTES – Parcial 2 Práctico
### ISIS-2611 | Para consultar rápido durante el parcial

---

## 1. IDENTIFICA EL PROBLEMA EN 10 SEGUNDOS

```
¿Hay etiqueta Y?
├── SÍ → SUPERVISADO
│   ├── Y es número continuo → REGRESIÓN
│   ├── Y tiene 2 categorías → BINARIO
│   └── Y tiene N > 2 categ. → MULTICLASE
└── NO → NO SUPERVISADO → CLUSTERING
    ├── Grupos esféricos, sin outliers → K-Means
    ├── Forma libre, hay outliers     → DBSCAN
    └── Quiero dendrograma / jerarquía → Jerárquico
```

**Palabras clave del enunciado:**

| Si dice... | Tipo |
|-----------|------|
| "predecir precio", "estimar cuánto" | Regresión |
| "sí/no", "fraude/normal", "spam/no spam" | Binario |
| "clasificar en N tipos", "categoría" | Multiclase |
| "agrupar", "segmentar", "sin etiquetas" | Clustering |
| "no pertenece a ningún grupo", "outliers" | DBSCAN |

---
## 2. ACTIVACIÓN + PÉRDIDA + NEURONAS DE SALIDA

| Problema | Neuronas salida | Activación salida | Loss |
|----------|----------------|-------------------|------|
| Regresión | 1 | **ninguna** | `mse` |
| Binario | 1 | **`sigmoid`** | `binary_crossentropy` |
| Multiclase (etiquetas enteras 0,1,2…) | N | **`softmax`** | `sparse_categorical_crossentropy` |
| Multiclase (etiquetas one-hot) | N | **`softmax`** | `categorical_crossentropy` |
| Capas **ocultas** (siempre) | cualquiera | **`relu`** | — |

### ❌ Errores más comunes de activación/pérdida
```
relu  en salida binaria      → sigmoid
relu  en salida multiclase   → softmax
sigmoid para N > 2 clases    → softmax
mse   para clasificación     → crossentropy
binary_crossentropy para N clases → sparse_categorical_crossentropy
categorical_crossentropy con etiquetas enteras → sparse_categorical_crossentropy
```

---
## 3. ORDEN CORRECTO DEL PIPELINE (obligatorio memorizar)

```
PASO 1 → train_test_split(X, y, stratify=y)
PASO 2 → imputer.fit(X_train)       → imputer.transform(X_test)
PASO 3 → scaler.fit(X_train)        → scaler.transform(X_test)
PASO 4 → SMOTE.fit_resample(X_train, y_train)   ← SOLO en train
PASO 5 → tokenizer.fit_on_texts(X_train_texto)  ← SOLO en train
```

### Regla de oro del Data Leakage
> **El test set NO puede influir en ningún objeto que `.fit()`**

```python
# ❌ MAL
scaler.fit(X)                          # ve el test
X_train, X_test = split(scaler.transform(X), y)

# ✅ BIEN
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y)
scaler.fit(X_train)                    # solo ve train
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)
```

### ¿Cuándo usar `stratify=y`?
> Siempre que el problema sea **clasificación**. Sin excepción.

---
## 4. PREPROCESAMIENTO: QUÉ USAR EN CADA CASO

### Variables numéricas → escalar
| Situación | Scaler |
|-----------|--------|
| Datos normales, sin outliers grandes | `StandardScaler` |
| Datos con outliers (ej: ingresos, precios) | `RobustScaler` |
| Quiero rango fijo [0,1] | `MinMaxScaler` |
| Variables binarias (0/1) | **No escalar** |

### Variables categóricas → encodear
| Tipo | Ejemplo | Encoder |
|------|---------|--------|
| **Nominal** (sin orden) | ciudad, color, marca | `OneHotEncoder` |
| **Ordinal** (con orden) | bajo<medio<alto, XS<S<M<L | `OrdinalEncoder(categories=[[...]])` |
| **Binaria** texto | sí/no | `.map({'SI':1,'NO':0})` |

```
❌ OrdinalEncoder en nominal  → inventa orden falso
❌ OneHotEncoder en ordinal   → pierde el orden
❌ Escalar después de OHE     → las columnas binarias se distorsionan
```

---
## 5. PLANTILLA MLP CORRECTA

```python
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Binario
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(n_features,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')          # ← binario
])
model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Multiclase (N clases, etiquetas enteras)
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(n_features,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(N, activation='softmax')          # ← N neuronas
])
model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# EarlyStopping (siempre usarlo)
es = callbacks.EarlyStopping(monitor='val_loss', patience=10,
                              restore_best_weights=True)
model.fit(X_train, y_train, epochs=200,
          validation_split=0.2, callbacks=[es])

# Predicción
y_prob = model.predict(X_test)          # probabilidades
y_pred_bin   = (y_prob.flatten() > 0.5).astype(int)   # binario
y_pred_multi = np.argmax(y_prob, axis=1)               # multiclase
```

---
## 6. PLANTILLA CNN CORRECTA

```python
from tensorflow.keras import layers, models

# SIEMPRE normalizar primero
train_images = train_images / 255.0
test_images  = test_images  / 255.0

# Si son escala de grises: añadir canal
# X_train = X_train.reshape(-1, 28, 28, 1)

model = models.Sequential([
    # Bloque convolucional 1
    layers.Conv2D(32, (3,3), padding='same', activation='relu',
                  input_shape=(H, W, C)),   # ← C=3 RGB, C=1 grises
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.2),

    # Bloque convolucional 2
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Clasificador
    layers.Flatten(),                        # ← OBLIGATORIO antes de Dense
    layers.Dense(128, activation='relu'),
    layers.Dense(N, activation='softmax')    # ← N clases
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
```

### ❌ Errores críticos de CNN
```
Dense antes de Flatten           → CRASH de shape
input_shape=(28,28)              → falta canal → debe ser (28,28,1)
imágenes sin dividir entre 255   → gradientes inestables
sigmoid en salida de N clases    → softmax
binary_crossentropy para N clases → sparse_categorical_crossentropy
```

---
## 7. PLANTILLA NLP + EMBEDDING CORRECTA

```python
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB = 10000
LEN   = 200

# 1. Split PRIMERO
X_tr, X_te, y_tr, y_te = train_test_split(textos, y, stratify=y)

# 2. Tokenizer solo en train
tok = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tok.fit_on_texts(X_tr)                         # ← solo train

X_tr = pad_sequences(tok.texts_to_sequences(X_tr), maxlen=LEN, padding='pre')
X_te = pad_sequences(tok.texts_to_sequences(X_te), maxlen=LEN, padding='pre')

# 3. Modelo
model = keras.Sequential([
    layers.Embedding(VOCAB, 64, input_length=LEN),   # ← convierte índices en vectores
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')            # binario
])
model.compile('adam', 'binary_crossentropy', ['accuracy'])
```

### ❌ Errores críticos de NLP
```
Texto crudo directo a la red      → tokenizar + Embedding primero
LSTM sin capa Embedding           → los índices no tienen significado
tok.fit_on_texts(train + test)    → leakage
SimpleRNN en secuencias largas    → LSTM o GRU (vanishing gradient)
```

---
## 8. LSTM / GRU – REGLAS DE return_sequences

```
Si después del LSTM viene OTRO LSTM/GRU  → return_sequences=True
Si después del LSTM viene Dense          → return_sequences=False  (default)

Ejemplo apilado:
  LSTM(64, return_sequences=True)   ← hay otro LSTM abajo
  LSTM(32, return_sequences=False)  ← el último, va a Dense
  Dense(1, activation='sigmoid')
```

### Series de tiempo con LSTM
```python
# Los datos deben ser 3D: (muestras, pasos_de_tiempo, features)
X = X.reshape(X.shape[0], X.shape[1], 1)   # (N, 30, 1) si 1 sola variable

model = keras.Sequential([
    layers.LSTM(64, input_shape=(30, 1)),
    layers.Dense(1)                          # regresión → sin activación
])
model.compile('adam', 'mse')
```

### Bidireccional: cuándo SÍ y cuándo NO
```
✅ Clasificación de texto (tienes toda la oración)    → Bidirectional(LSTM(64))
❌ Generación de texto (solo ves el pasado)           → LSTM normal
❌ Predicción de series de tiempo (el futuro no existe) → LSTM normal
```

---
## 9. CLUSTERING – PLANTILLA CORRECTA

```python
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# SIEMPRE: solo columnas numéricas
X = df[['col_num1', 'col_num2', 'col_num3']]

# SIEMPRE: escalar antes
scaler = RobustScaler()
X_sc = scaler.fit_transform(X)

# Elegir k con codo + silhouette
inertias, scores = [], []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sc)
    inertias.append(km.inertia_)
    scores.append(silhouette_score(X_sc, labels))

# Mejor k según gráficas:
km_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
labels = km_final.fit_predict(X_sc)
print(f'Silhouette: {silhouette_score(X_sc, labels):.3f}')
```

### ❌ Errores críticos de Clustering
```
Datos sin escalar                    → variables grandes dominan la distancia
Variables categóricas en KMeans      → KMeans solo acepta numéricas
k elegido sin codo + silhouette      → arbitrario
accuracy_score para evaluar cluster  → no tiene etiquetas reales → silhouette
Labels de KMeans comparados directamente entre ejecuciones → son arbitrarios
```

---
## 10. NAIVE BAYES

### Tipos de Naive Bayes
```
GaussianNB     → features CONTINUAS (supone distribución Normal)
MultinomialNB  → CONTEOS de palabras, frecuencias
BernoulliNB    → features BINARIAS (presencia/ausencia)
```

### Generativo vs Discriminativo
```
GENERATIVO   (Naive Bayes, GAN): aprende P(X,Y) → puede GENERAR nuevos X
DISCRIMINATIVO (Regresión logística, SVM, Redes): aprende P(Y|X) → solo clasifica

Naive Bayes ES generativo porque aprende la distribución de cada feature
para cada clase (theta_ = medias, var_ = varianzas).
Con eso puede muestrear nuevos ejemplos: np.random.normal(media, std)
```

### Plantilla correcta
```python
from sklearn.naive_bayes import MultinomialNB   # para texto/conteos
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y)

clf = MultinomialNB(alpha=1.0)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
```

---
## 11. MÉTRICAS – CUÁNDO USAR CADA UNA

| Situación | Métricas correctas | ❌ Nunca |
|-----------|-------------------|----------|
| Clasificación balanceada | accuracy, F1 | — |
| Dataset desbalanceado | ROC-AUC, recall clase positiva, F1 macro | accuracy |
| Medicina (no perder positivos) | **Recall** de la clase enferma | accuracy |
| Spam (no perder correos buenos) | **Precision** de la clase spam | — |
| Regresión | MSE, MAE, R² | accuracy, F1 |
| Clustering | Silhouette, Inercia | accuracy |

### Matriz de confusión (leer así)
```
                 Predicho NEG    Predicho POS
  Real NEG   [[    TN          ,    FP (falsa alarma)  ]]
  Real POS   [[    FN (fallo!) ,    TP (detección ✅)  ]]

Precision = TP / (TP + FP)   → de los que predije POS, ¿cuántos lo eran?
Recall    = TP / (TP + FN)   → de todos los POS reales, ¿cuántos detecté?
F1        = media armónica entre Precision y Recall
```

### F1 macro vs weighted
```
macro    → todas las clases pesan IGUAL  ← usar con desbalance
weighted → pondera por nº de muestras   ← puede ocultar clases raras
```

### Evaluación final
```python
from sklearn.metrics import classification_report, roc_auc_score
# Esto lo muestra todo de una vez:
print(classification_report(y_test, y_pred, target_names=['clase_0','clase_1']))
```

---
## 12. RESUMEN VISUAL: ARQUITECTURAS

```
MLP (datos tabulares)
────────────────────
  input (n,)  →  Dense+relu  →  Dropout  →  Dense+relu  →  Dense+activación_salida


CNN (imágenes)
──────────────
  imagen ÷255 (H,W,C)
    → Conv2D+relu  → MaxPooling
    → Conv2D+relu  → MaxPooling
    → FLATTEN                   ← OBLIGATORIO
    → Dense+relu
    → Dense+softmax/sigmoid


NLP / Texto
───────────
  texto_crudo
    → Tokenizer.fit(train) → texts_to_sequences → pad_sequences
    → Embedding(vocab, dim, maxlen)
    → LSTM / GRU / GlobalAveragePooling1D
    → Dense+softmax/sigmoid


Series de tiempo
────────────────
  X.reshape(N, pasos, 1)
    → LSTM(64, input_shape=(pasos, 1))
    → Dense(1)  ← sin activación (regresión)
    compile con loss='mse'
```

---
## 13. CHECKLIST ANTES DE ENTREGAR

### Arquitectura
- [ ] ¿La activación de salida corresponde al tipo de problema?
- [ ] ¿El número de neuronas de salida es correcto? (1 para binario/regresión, N para multiclase)
- [ ] ¿La función de pérdida corresponde a la activación?
- [ ] ¿Las capas ocultas usan `relu`?
- [ ] CNN: ¿hay `Flatten()` antes de `Dense`?
- [ ] CNN: ¿`input_shape` incluye el canal de color? `(H, W, C)`
- [ ] CNN: ¿las imágenes están divididas entre 255?
- [ ] LSTM apilados: ¿el primero tiene `return_sequences=True`?
- [ ] NLP: ¿hay capa `Embedding` antes del LSTM?

### Pipeline de datos
- [ ] ¿El `train_test_split` va **antes** de cualquier `.fit()`?
- [ ] ¿Usé `stratify=y`?
- [ ] ¿`scaler.fit()` solo en `X_train`?
- [ ] ¿`imputer.fit()` solo en `X_train`?
- [ ] ¿`SMOTE` solo en `X_train`?
- [ ] ¿`Tokenizer.fit_on_texts()` solo con `X_train`?
- [ ] ¿Variables nominales con `OneHotEncoder` (no `OrdinalEncoder`)?
- [ ] ¿Clustering: datos escalados + solo numéricas?

### Evaluación
- [ ] ¿Estoy usando la métrica correcta para este tipo de problema?
- [ ] Si el dataset está desbalanceado, ¿evité usar solo accuracy?
- [ ] Para clasificación, ¿miré el `classification_report` completo?
- [ ] ¿`model.predict()` → apliqué `argmax` o umbral 0.5 para obtener las clases?

---
## 14. ERRORES MÁS TRAMPOSOS (los que más duelen)

```python
# ❌ 1. Dense antes de Flatten en CNN
layers.MaxPooling2D((2,2)),
layers.Dense(64, activation='relu'),   # CRASH: tensor 3D a Dense
# ✅ Agregar Flatten() antes

# ❌ 2. input_shape sin canal
layers.Conv2D(32, (3,3), input_shape=(28, 28))     # falta canal
# ✅
layers.Conv2D(32, (3,3), input_shape=(28, 28, 1))  # escala de grises
layers.Conv2D(32, (3,3), input_shape=(32, 32, 3))  # RGB

# ❌ 3. Learning rate demasiado alto → NaN
keras.optimizers.Adam(learning_rate=5.0)
# ✅
keras.optimizers.Adam(learning_rate=0.001)

# ❌ 4. LSTM apilados sin return_sequences=True
layers.LSTM(64),           # devuelve (batch, 64) — 2D
layers.LSTM(32),           # espera (batch, steps, 64) — 3D → ERROR
# ✅
layers.LSTM(64, return_sequences=True),
layers.LSTM(32),

# ❌ 5. Scaler antes del split
scaler.fit(X_todo)                    # leakage
# ✅
X_train, X_test = split(...)
scaler.fit(X_train)

# ❌ 6. SMOTE antes del split
X_res, y_res = smote.fit_resample(X, y)   # sintéticos en test
X_train, X_test = split(X_res, y_res)
# ✅
X_train, X_test = split(X, y)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# ❌ 7. OrdinalEncoder en nominal
OrdinalEncoder().fit_transform(df[['ciudad']])
# ✅
OneHotEncoder(handle_unknown='ignore').fit_transform(df[['ciudad']])

# ❌ 8. Accuracy en dataset desbalanceado
accuracy_score(y_test, y_pred)         # puede ser 98% y el modelo ser inútil
# ✅
print(classification_report(y_test, y_pred))
roc_auc_score(y_test, y_prob)

# ❌ 9. model.predict() sin convertir a clases
accuracy_score(y_test, model.predict(X_test))      # predict devuelve probs
# ✅
y_prob = model.predict(X_test)
y_pred = (y_prob.flatten() > 0.5).astype(int)      # binario
y_pred = np.argmax(y_prob, axis=1)                  # multiclase

# ❌ 10. SimpleRNN en texto largo
layers.SimpleRNN(64)     # olvida el inicio de la secuencia (vanishing gradient)
# ✅
layers.LSTM(64)          # o layers.GRU(64)
```

---
## 15. CÓDIGO DE REFERENCIA RÁPIDA (copy-paste)

```python
# ── Imports básicos ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, silhouette_score, confusion_matrix
from sklearn.cluster import KMeans, DBSCAN
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from imblearn.over_sampling import SMOTE
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ── EarlyStopping ────────────────────────────────────────────────────────────
es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# ── Evaluación completa (clasificación) ─────────────────────────────────────
y_prob = model.predict(X_test).flatten()
y_pred = (y_prob > 0.5).astype(int)    # binario
# y_pred = np.argmax(model.predict(X_test), axis=1)  # multiclase
print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.show()

# ── Silhouette para clustering ───────────────────────────────────────────────
print(f'Silhouette: {silhouette_score(X_scaled, labels):.3f}')
```